In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Modèle chargé !")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Acer\Desktop\RAG PROJECT\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modèle chargé !


In [3]:
text = "Can I still join the course after the start date?"
vector = model.encode(text)

print("Taille du vecteur:", len(vector))
print("Les 5 premiers nombres:", vector[:5])

Taille du vecteur: 384
Les 5 premiers nombres: [ 0.0213904  -0.07397993  0.00142068  0.02138164  0.0245113 ]


In [4]:
from sentence_transformers import util

question1 = "Can I still join the course after the start date?"
question2 = "Is it possible to enroll late?"
question3 = "How do I cook pasta?"

vec1 = model.encode(question1)
vec2 = model.encode(question2)
vec3 = model.encode(question3)

sim_12 = util.cos_sim(vec1, vec2)
sim_13 = util.cos_sim(vec1, vec3)

print(f"Similarité Q1 vs Q2 (même sens): {sim_12.item():.4f}")
print(f"Similarité Q1 vs Q3 (sens différent): {sim_13.item():.4f}")

Similarité Q1 vs Q2 (même sens): 0.5209
Similarité Q1 vs Q3 (sens différent): -0.0298


In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
docs_response = requests.get(docs_url)
courses_raw = docs_response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_data = course_response.json()
    documents.extend(course_data)

docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"✅ {len(docs_llm)} documents chargés !")

✅ 85 documents chargés !


In [6]:
import numpy as np

embeddings = []

for doc in docs_llm:
    text = doc["question"] + " " + doc["answer"]
    vector = model.encode(text)
    embeddings.append(vector)

embeddings = np.array(embeddings)
print(f"✅ Shape des embeddings: {embeddings.shape}")

✅ Shape des embeddings: (85, 384)


In [7]:
from sentence_transformers import util

def vector_search(question, top_k=5):
    query_vector = model.encode(question)
    
    scores = util.cos_sim(query_vector, embeddings)[0]
    
    top_indices = scores.argsort(descending=True)[:top_k]
    
    results = []
    for idx in top_indices:
        doc = docs_llm[idx]
        results.append({
            "score": scores[idx].item(),
            "question": doc["question"],
            "answer": doc["answer"]
        })
    
    return results

In [8]:
from sentence_transformers import util

def vector_search(question, top_k=5):
    query_vector = model.encode(question)
    
    scores = util.cos_sim(query_vector, embeddings)[0]
    
    top_indices = scores.argsort(descending=True)[:top_k]
    
    results = []
    for idx in top_indices:
        doc = docs_llm[idx]
        results.append({
            "score": scores[idx].item(),
            "question": doc["question"],
            "answer": doc["answer"]
        })
    
    return results

In [9]:
results = vector_search("Is it possible to enroll late?")

for r in results:
    print(f"Score: {r['score']:.4f}")
    print(f"Question: {r['question']}")
    print("---")

Score: 0.3728
Question: I just discovered the course. Can I still join?
---
Score: 0.3490
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
---
Score: 0.3026
Question: Certificate: Can I follow the course in a self-paced mode and get a certificate?
---
Score: 0.2989
Question: When will the course be offered next?
---
Score: 0.2555
Question: Can I use Bluesky for learning in public credits?
---


In [10]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.
Use only the facts from the CONTEXT when answering the QUESTION.
If the answer is not found in the context, respond with "I don't know."
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    return "\n".join(lines).strip()

def rag_vector(question):
    search_results = vector_search(question)
    context = build_context(search_results)
    
    prompt = f"""
QUESTION: {question}

CONTEXT:
{context}
""".strip()

    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [11]:
answer = rag_vector("Is it possible to enroll late?")
print(answer)

Based on the given context, it appears you have two separate questions:

1. Is it possible to enroll late?
   Since late enrollment does not have a specific section, one has to infer this from given context.

   In the context, the registration process is mentioned, and the context mentions 'the form is open' to submit homework. This form being 'open' suggests an open period for enrollment for the course.


In [1]:
from minsearch import Index

# Index keyword
keyword_index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)
keyword_index.fit(docs_llm)

def keyword_search(question):
    return keyword_index.search(
        question,
        boost_dict={"question": 2.0, "section": 0.5},
        num_results=3
    )

# Comparer
question = "Is it possible to enroll late?"

print("=== KEYWORD SEARCH ===")
for r in keyword_search(question):
    print(r["question"])

print("\n=== VECTOR SEARCH ===")
for r in vector_search(question, top_k=3):
    print(r["question"])

NameError: name 'docs_llm' is not defined